In [ ]:
import numpy as np

def moment_to_stress(M_mid, axle_pos_time, Pi_full, pos_last,
                     l,
                     I, y, A,
                     mu=0.71,
                     axial_mode="any_on_bridge",
                     axial_sign=-1.0,
                     to_pa=False):
    """
    This function converts midspan moment time histories into stress time histories

    outputs:
        sigma_total as ndarray, sigma_b as ndarray (component of stress due to bending), sigma_axial (component of stress due to friction of wheels)
    """
    M_mid = np.asarray(M_mid, dtype=float)
    Pi_full = np.asarray(Pi_full, dtype=float)
    axle_pos_time = np.asarray(axle_pos_time, dtype=float)

    n_steps = M_mid.shape[0]
    n_axles = Pi_full.shape[0]

    if axle_pos_time.shape[0] != n_steps:
        raise ValueError("axle_pos_time must have same n_steps as M_mid.")

    sigma_b = (M_mid * y) / I   # flexure formula for the bending part

    sigma_axial = np.zeros(n_steps, dtype=float)

    for i in range(n_steps):
        axle_pos = axle_pos_time[i, :]  
        if axial_mode == "any_on_bridge":
            on_mask = (axle_pos >= 0.0) & (axle_pos <= l)
        elif axial_mode == "after_midspan":
            on_mask = axle_pos >= (l / 2.0)
        else:
            raise ValueError("axial_mode must be 'any_on_bridge' or 'after_midspan'.")

        if not on_mask.any():
            sigma_axial[i] = 0.0
        else:
            F_axial_kgf = mu * np.sum(Pi_full[on_mask])
            sigma_axial[i] = axial_sign * (F_axial_kgf / A)

    sigma_total = sigma_b + sigma_axial

    if to_pa:
        sigma_b = sigma_b * 9.81
        sigma_axial = sigma_axial * 9.81
        sigma_total = sigma_total * 9.81

    return sigma_total, sigma_b, sigma_axial

